# Model

In [ ]:
import os
import sys
import time
import json
import shutil
import hashlib
import warnings
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import cv2

# Comprehensive Metrics
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score,
    top_k_accuracy_score
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from sklearn.calibration import calibration_curve

# Explainability & Segmentation Libraries
from lime import lime_image
from skimage.segmentation import mark_boundaries, slic
import shap
import kagglehub

warnings.filterwarnings('ignore', category=UserWarning)


# ==========================================
# 0. Virtual Data Crawler & Path Parser
# ==========================================
def get_raw_dataset_paths():
    """Checks for Kaggle environment and dynamically handles unmounted datasets"""
    raw_paths = []
    datasets = [
        "andrewmvd/lung-and-colon-cancer-histopathological-images",
        "ambarish/breakhis",
        "mehradaria/leukemia",
        "prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed",
        "ashenafifasilkebede/dataset", # Oral
        "nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone",
        "masoudnickparvar/brain-tumor-mri-dataset" # Brain
    ]

    kaggle_input = '/kaggle/input'
    if os.path.exists(kaggle_input) and len(os.listdir(kaggle_input)) > 0:
        print(f"[Data Pipeline] Kaggle environment detected. Scanning mounted datasets in {kaggle_input}...")
        mounted_dirs = [os.path.join(kaggle_input, d) for d in os.listdir(kaggle_input)]
        raw_paths.extend(mounted_dirs)
        
        brain_mounted = any('brain' in d.lower() or 'mri' in d.lower() or 'masoudnickparvar' in d.lower() for d in mounted_dirs)
        
        if not brain_mounted:
            print("[Data Pipeline] Brain Cancer MRI dataset not found in /kaggle/input mounts!")
            print("[Data Pipeline] Attempting dynamic download via KaggleHub...")
            try:
                raw_paths.append(kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset"))
            except Exception as e:
                print(f"[Warning] Failed to download Brain MRI dataset dynamically: {e}")
    else:
        print("[Data Pipeline] Local environment detected. Downloading all via KaggleHub...")
        for ds in datasets:
            try:
                raw_paths.append(kagglehub.dataset_download(ds))
            except Exception as e:
                print(f"[Warning] Failed to download {ds}: {e}")

    if not raw_paths:
        raise RuntimeError("No datasets were downloaded and no local Kaggle input was found. Please check your internet/Kaggle credentials.")
    return raw_paths


def resolve_type_and_stage(path):
    """
    Bulletproof deterministic path mapping based on absolute file paths.
    Using an elif ladder and dataset slugs guarantees no cross-contamination.
    """
    path_lower = path.lower().replace('\\', '/')
    parts = path_lower.split('/')
    if len(parts) < 2: return None, None
    parent = parts[-2]

    # 1. Breast (BreaKHis_v1)
    if 'breakhis' in path_lower or 'breast' in path_lower or 'ambarish' in path_lower:
        if 'benign' in path_lower: return 'Breast', 'Benign'
        if 'malignant' in path_lower: return 'Breast', 'Malignant'

    # 2. Kidney (CT-Kidney) 
    elif 'kidney' in path_lower or 'ct-kidney' in path_lower or 'nazmul' in path_lower:
        if 'tumor' in parent: return 'Kidney', 'Tumor'
        if 'cyst' in parent: return 'Kidney', 'Cyst'
        if 'stone' in parent: return 'Kidney', 'Stone'
        if 'normal' in parent: return 'Kidney', 'Normal'

    # 3. Brain (Brain Tumor MRI)
    elif 'brain' in path_lower or 'mri' in path_lower or 'masoudnickparvar' in path_lower or 'glioma' in path_lower or 'meningioma' in path_lower:
        if 'glioma' in path_lower: return 'Brain', 'Glioma'
        if 'meningioma' in path_lower: return 'Brain', 'Meningioma'
        if 'pituitary' in path_lower: return 'Brain', 'Pituitary_Tumor'
        if 'notumor' in path_lower or 'no_tumor' in path_lower: return 'Brain', 'Normal'

    # 4. Lung and Colon (LC25000)
    elif 'lung' in path_lower or 'colon' in path_lower or 'lc25000' in path_lower or 'andrewmvd' in path_lower:
        if 'colon_aca' in path_lower: return 'Colon', 'Adenocarcinoma'
        if 'colon_n' in path_lower: return 'Colon', 'Benign'
        if 'lung_aca' in path_lower: return 'Lung', 'Adenocarcinoma'
        if 'lung_scc' in path_lower: return 'Lung', 'Squamous_Cell'
        if 'lung_n' in path_lower: return 'Lung', 'Benign'

    # 5. Leukemia (ALL)
    elif 'leukemia' in path_lower or 'c-nmc' in path_lower or 'lymphoblastic' in path_lower or 'mehradaria' in path_lower:
        if 'benign' in path_lower or 'hem' in path_lower: return 'Leukemia', 'Benign'
        if 'early' in path_lower: return 'Leukemia', 'Malignant_Early'
        if 'pre' in path_lower and 'early' not in path_lower: return 'Leukemia', 'Malignant_Pre'
        if 'pro' in path_lower: return 'Leukemia', 'Malignant_Pro'
        if 'all' in path_lower: return 'Leukemia', 'Malignant_Lymphoblasts'

    # 6. Cervical (Sipakmed - nested .bmp folders)
    elif 'sipakmed' in path_lower or 'cervical' in path_lower or 'prahladmehandiratta' in path_lower:
        if 'dyskeratotic' in path_lower: return 'Cervical', 'Abnormal_Dyskeratotic'
        if 'koilocytotic' in path_lower: return 'Cervical', 'Abnormal_Koilocytotic'
        if 'metaplastic' in path_lower: return 'Cervical', 'Abnormal_Metaplastic'
        if 'parabasal' in path_lower: return 'Cervical', 'Abnormal_Parabasal'
        if 'superficial' in path_lower or 'normal' in path_lower or 'intermediate' in path_lower: return 'Cervical', 'Normal_Superficial'

    # 7. Oral 
    elif 'oscc' in path_lower or 'oral' in path_lower or 'histopathological' in path_lower or 'ashenafi' in path_lower:
        if 'oscc' in path_lower: return 'Oral', 'Malignant_OSCC'
        if 'normal' in path_lower: return 'Oral', 'Benign_Normal'

    return None, None


def extract_patient_id(path, cancer_type):
    """
    Best-effort patient/slide grouping key so that the same physical patient
    cannot appear in both train and val/test.
    """
    p = path.replace('\\', '/')
    fname = os.path.basename(p)
    stem = os.path.splitext(fname)[0]

    if cancer_type == 'Breast':
        segs = stem.split('-')
        if len(segs) >= 3:
            return f"Breast::{segs[1]}-{segs[2]}"

    if cancer_type == 'Leukemia':
        parent = os.path.basename(os.path.dirname(p))
        return f"Leukemia::{parent}::{stem.split('_')[0]}"

    if cancer_type in ('Lung', 'Colon'):
        base = ''.join(ch for ch in stem if not ch.isdigit())
        return f"{cancer_type}::{base}::{stem[:12]}"

    if cancer_type == 'Oral':
        return f"Oral::{stem.split('_')[0]}"

    if cancer_type == 'Cervical':
        return f"Cervical::{stem.split('_')[0]}"

    if cancer_type == 'Kidney':
        return f"Kidney::{stem.split('_')[0]}"

    if cancer_type == 'Brain':
        return f"Brain::{stem.split('_')[0]}"

    return f"{cancer_type}::{stem}"


# ==========================================
# 1. Configuration Class
# ==========================================
class Config:
    def __init__(self):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        # Directories
        base_working_dir = ''
        self.ckpt_dir = os.path.join(base_working_dir, 'models')
        self.output_dir = os.path.join(base_working_dir, 'outputs')
        self.checkpoint_path = os.path.join(self.ckpt_dir, 'CanceRX_ckpt.pth')

        os.makedirs(self.ckpt_dir, exist_ok=True)
        os.makedirs(self.output_dir, exist_ok=True)

        # Hyperparameters
        self.img_size = 224
        self.in_chans = 3
        self.num_types = None
        self.num_stages = None
        self.dim = 512
        self.depth = 4
        self.num_heads = 8
        self.num_experts = 4
        self.patch_size = 16

        self.epochs = 150
        self.batch_size = 64
        self.lr = 3e-4
        self.weight_decay = 1e-4

        # Split control (70% train, 15% validation, 15% test)
        self.train_fraction = 0.70
        self.val_fraction = 0.15
        self.test_fraction = 0.15
        self.split_seed = 42
        self.patient_aware_split = True

        # Complexity / benchmarking
        self.complexity_input_shapes = [(1, 3, 224, 224)]
        self.bench_batch_sizes = [1, 8, 32]
        self.bench_warmup_iters = 20
        self.bench_measure_iters = 100

        # XAI budgets
        self.lime_num_samples = 1000
        self.shap_nsamples = 200
        self.shap_segments = 32
        self.ig_steps = 64
        self.scorecam_topk = 32
        self.occlusion_patch = 32
        self.occlusion_stride = 8


# ==========================================
# 2. Custom Loss Class (OmniLoss)
# ==========================================
class OmniLoss(nn.Module):
    """Combines Cross Entropy with Focal modifier for imbalanced datasets."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()
    
# ==========================================
# 3. Virtual Hierarchical Dataset Class
# ==========================================
class HierarchicalCancerDataset(Dataset):
    STAIN_TARGET_MEANS = torch.tensor([0.7, 0.5, 0.6]).view(3, 1, 1)
    STAIN_TARGET_STDS = torch.tensor([0.1, 0.1, 0.15]).view(3, 1, 1)
    LAPLACIAN = torch.tensor(
        [[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=torch.float32
    ).view(1, 1, 3, 3).repeat(3, 1, 1, 1)

    def __init__(self, data_dirs, config, is_train=True, shared_index=None):
        self.config = config
        self.is_train = is_train

        if shared_index is not None:
            self.samples = shared_index['samples']
            self.types = shared_index['types']
            self.stages = shared_index['stages']
            self.type_to_idx = shared_index['type_to_idx']
            self.stage_to_idx = shared_index['stage_to_idx']
            config.num_types = len(self.types)
            config.num_stages = len(self.stages)
        else:
            self.samples = []
            types_set, stages_set = set(), set()

            print("[Dataset] Building virtual memory map of filesystem...")
            valid_exts = ('.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp')

            for data_dir in data_dirs:
                for root, _, files in os.walk(data_dir):
                    for file in sorted(files):
                        if not file.lower().endswith(valid_exts):
                            continue
                        path = os.path.join(root, file)
                        t, s = resolve_type_and_stage(path)
                        if t and s:
                            types_set.add(t)
                            full_stage_name = f"{t}_{s}"
                            stages_set.add(full_stage_name)
                            self.samples.append({
                                'path': path,
                                'type_name': t,
                                'stage_name': full_stage_name,
                                'patient_id': extract_patient_id(path, t)
                            })

            if len(self.samples) == 0:
                raise RuntimeError("No valid images successfully mapped. Verify directory paths.")

            self.samples.sort(key=lambda d: d['path'])
            self.types = sorted(list(types_set))
            self.stages = sorted(list(stages_set))
            self.type_to_idx = {t: i for i, t in enumerate(self.types)}
            self.stage_to_idx = {s: i for i, s in enumerate(self.stages)}

            config.num_types = len(self.types)
            config.num_stages = len(self.stages)

            print(f"[Dataset] Success! Mapped {len(self.samples)} real images natively.")
            print(f" -> Level 1 Types ({config.num_types}): {self.types}")
            print(f" -> Level 2 Stages ({config.num_stages}): {self.stages}")
            n_patients = len(set(d['patient_id'] for d in self.samples))
            print(f" -> Distinct grouping keys (patient/slide): {n_patients}")

        if self.is_train:
            self.base_transform = transforms.Compose([
                transforms.Resize((config.img_size, config.img_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
                transforms.ToTensor()
            ])
        else:
            self.base_transform = transforms.Compose([
                transforms.Resize((config.img_size, config.img_size)),
                transforms.ToTensor()
            ])

    def export_index(self):
        return {
            'samples': self.samples,
            'types': self.types,
            'stages': self.stages,
            'type_to_idx': self.type_to_idx,
            'stage_to_idx': self.stage_to_idx,
        }

    @staticmethod
    def stain_normalization(img_tensor):
        tm = HierarchicalCancerDataset.STAIN_TARGET_MEANS.to(img_tensor.device)
        ts = HierarchicalCancerDataset.STAIN_TARGET_STDS.to(img_tensor.device)
        img_mean = img_tensor.mean(dim=(1, 2), keepdim=True)
        img_std = img_tensor.std(dim=(1, 2), keepdim=True) + 1e-6
        normalized = (img_tensor - img_mean) / img_std
        return torch.clamp(normalized * ts + tm, 0, 1)

    @staticmethod
    def morphological_edge_enhancement(img_tensor):
        lap = HierarchicalCancerDataset.LAPLACIAN.to(img_tensor.device)
        img_padded = F.pad(img_tensor.unsqueeze(0), (1, 1, 1, 1), mode='reflect')
        edges = F.conv2d(img_padded, lap, groups=3).squeeze(0)
        return torch.clamp(img_tensor + 0.3 * edges, 0, 1)

    @staticmethod
    def preprocess_from_hwc_float(img_hwc):
        """
        Replays the EXACT training preprocessing on a HWC float array in [0,1].
        """
        t = torch.from_numpy(np.ascontiguousarray(img_hwc)).permute(2, 0, 1).float()
        t = torch.clamp(t, 0, 1)
        t = HierarchicalCancerDataset.stain_normalization(t)
        t = HierarchicalCancerDataset.morphological_edge_enhancement(t)
        return t

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        raw_img = Image.open(item['path']).convert('RGB')

        img_tensor = self.base_transform(raw_img)
        img_tensor = self.stain_normalization(img_tensor)
        img_tensor = self.morphological_edge_enhancement(img_tensor)

        type_idx = self.type_to_idx[item['type_name']]
        stage_idx = self.stage_to_idx[item['stage_name']]

        return img_tensor, torch.tensor(type_idx, dtype=torch.long), torch.tensor(stage_idx, dtype=torch.long)


# ==========================================
# 4. Anti-Collapse CanceRX Architecture
# ==========================================
class CMPA(nn.Module):
    """Cosine-Modulated Phase Attention"""
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.phase_net = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(0.1)

        # Zero-initialize the Phase Net. 
        # This keeps the gate fully "open" at step 1 so gradients aren't chaotically blocked.
        nn.init.constant_(self.phase_net.weight, 0.0)
        nn.init.constant_(self.phase_net.bias, 0.0)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        phases = self.phase_net(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        phase_diff = phases.mean(dim=-1, keepdim=True) - phases.mean(dim=-1, keepdim=True).transpose(-2, -1)
        interference = torch.cos(phase_diff) 
        
        attn = attn * interference 
        attn = attn.softmax(dim=-1)

        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out), attn


class TPR(nn.Module):
    """TopologicalPathwayRouting"""
    def __init__(self, dim, num_experts):
        super().__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))
            for _ in range(num_experts)
        ])
        self.router = nn.Sequential(nn.Linear(dim, num_experts), nn.Softmax(dim=-1))

        nn.init.constant_(self.router[0].weight, 0.0)
        nn.init.constant_(self.router[0].bias, 0.0)

    def forward(self, x):
        routes = self.router(x)
        out = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            out = out + routes[:, :, i:i + 1] * expert(x)
        return out, routes


class TPGBlock(nn.Module):
    """Topological Phased Gated Block"""
    def __init__(self, dim, num_heads, num_experts):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = CMPA(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.tpr = TPR(dim, num_experts)

    def forward(self, x):
        attn_out, attn_map = self.attn(self.norm1(x))
        x = x + attn_out
        tpr_out, route_map = self.tpr(self.norm2(x))
        x = x + tpr_out
        return x, attn_map, route_map


class CanceRX(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.patch_size = config.patch_size
        self.img_size = config.img_size
        self.grid = config.img_size // self.patch_size
        self.patch_embed = nn.Conv2d(
            config.in_chans, config.dim,
            kernel_size=self.patch_size, stride=self.patch_size
        )
        self.num_patches = self.grid ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, config.dim) * 0.02)

        self.layers = nn.ModuleList([
            TPGBlock(config.dim, config.num_heads, config.num_experts)
            for _ in range(config.depth)
        ])

        self.norm = nn.LayerNorm(config.dim)
        self.head_type = nn.Linear(config.dim, config.num_types)
        self.head_stage = nn.Linear(config.dim, config.num_stages)

        self._cam_activations = None
        self._cam_gradients = None
        self._cam_handles = []

    def _fwd_hook(self, module, inp, out):
        self._cam_activations = out
        if out.requires_grad:
            out.register_hook(self._save_grad)

    def _save_grad(self, grad):
        self._cam_gradients = grad

    def enable_cam_hooks(self):
        self.disable_cam_hooks()
        h = self.patch_embed.register_forward_hook(self._fwd_hook)
        self._cam_handles.append(h)

    def disable_cam_hooks(self):
        for h in self._cam_handles:
            h.remove()
        self._cam_handles = []
        self._cam_activations = None
        self._cam_gradients = None

    @property
    def cam_activations(self):
        return self._cam_activations

    @property
    def cam_gradients(self):
        return self._cam_gradients

    def forward(self, x, return_internals=False):
        feat_map = self.patch_embed(x)                      
        x = feat_map.flatten(2).transpose(1, 2)              
        x = x + self.pos_embed

        attns, routes = [], []
        for layer in self.layers:
            x, attn_map, route_map = layer(x)
            if return_internals:
                attns.append(attn_map)
                routes.append(route_map)

        feats = self.norm(x.mean(dim=1))

        logits_type = self.head_type(feats)
        logits_stage = self.head_stage(feats)

        if return_internals:
            return logits_type, logits_stage, attns, routes, feats
        return logits_type, logits_stage, None, None, feats


# ==========================================
# 4b. Model Complexity Analyzer
# ==========================================
class ComplexityAnalyzer:
    def __init__(self, model, config):
        self.model = model
        self.config = config

    def parameter_report(self):
        total = sum(p.numel() for p in self.model.parameters())
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        non_trainable = total - trainable
        buffers = sum(b.numel() for b in self.model.buffers())

        per_module = {}
        for name, mod in self.model.named_children():
            per_module[name] = sum(p.numel() for p in mod.parameters())

        return {
            'total_parameters': total,
            'trainable_parameters': trainable,
            'non_trainable_parameters': non_trainable,
            'buffer_elements': buffers,
            'per_top_level_module': per_module,
        }

    def size_report(self):
        param_bytes = sum(p.numel() * p.element_size() for p in self.model.parameters())
        buffer_bytes = sum(b.numel() * b.element_size() for b in self.model.buffers())
        total_bytes = param_bytes + buffer_bytes
        n = sum(p.numel() for p in self.model.parameters())
        return {
            'param_size_MB': param_bytes / (1024 ** 2),
            'buffer_size_MB': buffer_bytes / (1024 ** 2),
            'total_size_MB': total_bytes / (1024 ** 2),
            'size_if_fp32_MB': n * 4 / (1024 ** 2),
            'size_if_fp16_MB': n * 2 / (1024 ** 2),
            'size_if_int8_MB': n * 1 / (1024 ** 2),
        }

    def macs_report(self, input_shape=(1, 3, 224, 224)):
        cfg = self.config
        B, C, H, W = input_shape
        g = cfg.img_size // cfg.patch_size
        N = g * g
        D = cfg.dim
        heads = cfg.num_heads
        hd = D // heads

        macs = 0
        breakdown = {}

        pe = N * (C * cfg.patch_size * cfg.patch_size) * D
        macs += pe
        breakdown['patch_embed'] = pe

        qkv = N * D * (3 * D)
        phase = N * D * D
        attn_qk = heads * N * N * hd
        attn_av = heads * N * N * hd
        proj = N * D * D
        router = N * D * cfg.num_experts
        expert_one = N * D * (2 * D) + N * (2 * D) * D
        experts = cfg.num_experts * expert_one

        per_block = qkv + phase + attn_qk + attn_av + proj + router + experts
        macs += cfg.depth * per_block

        breakdown['per_block_qkv'] = qkv
        breakdown['per_block_phase_net'] = phase
        breakdown['per_block_attn_QK'] = attn_qk
        breakdown['per_block_attn_AV'] = attn_av
        breakdown['per_block_out_proj'] = proj
        breakdown['per_block_moe_router'] = router
        breakdown['per_block_moe_experts_dense'] = experts
        breakdown['per_block_total'] = per_block
        breakdown['all_blocks_total'] = cfg.depth * per_block

        heads_macs = D * cfg.num_types + D * cfg.num_stages
        macs += heads_macs
        breakdown['classifier_heads'] = heads_macs

        macs *= B

        return {
            'input_shape': list(input_shape),
            'MACs': macs,
            'GMACs': macs / 1e9,
            'FLOPs': 2 * macs,
            'GFLOPs': 2 * macs / 1e9,
            'breakdown_MACs': breakdown,
            'note': 'FLOPs = 2 x MACs (multiply + accumulate). Dense MoE evaluates all experts.'
        }

    def run(self, out_dir):
        os.makedirs(out_dir, exist_ok=True)
        report = {
            'parameters': self.parameter_report(),
            'model_size': self.size_report(),
            'complexity': [self.macs_report(s) for s in self.config.complexity_input_shapes],
        }

        lines = ["=" * 74, "MODEL COMPLEXITY REPORT", "=" * 74]
        p = report['parameters']
        lines.append(f"Total parameters        : {p['total_parameters']:,}")
        lines.append(f"Trainable parameters    : {p['trainable_parameters']:,}")
        lines.append(f"Non-trainable parameters: {p['non_trainable_parameters']:,}")
        lines.append(f"Buffer elements         : {p['buffer_elements']:,}")
        lines.append("")
        lines.append("Parameters by top-level module:")
        for k, v in p['per_top_level_module'].items():
            lines.append(f"  {k:<16s} {v:>14,}")
        lines.append("")

        s = report['model_size']
        lines.append(f"Model size (actual dtype): {s['total_size_MB']:.3f} MB")
        lines.append(f"  params                 : {s['param_size_MB']:.3f} MB")
        lines.append(f"  buffers                : {s['buffer_size_MB']:.3f} MB")
        lines.append(f"Hypothetical fp32        : {s['size_if_fp32_MB']:.3f} MB")
        lines.append(f"Hypothetical fp16        : {s['size_if_fp16_MB']:.3f} MB")
        lines.append(f"Hypothetical int8        : {s['size_if_int8_MB']:.3f} MB")
        lines.append("")

        for c in report['complexity']:
            lines.append(f"Input {c['input_shape']}")
            lines.append(f"  GMACs  : {c['GMACs']:.4f}")
            lines.append(f"  GFLOPs : {c['GFLOPs']:.4f}")
            lines.append("  MAC breakdown:")
            for k, v in c['breakdown_MACs'].items():
                lines.append(f"    {k:<32s} {v/1e9:>10.5f} GMAC")
            lines.append("")
        lines.append("=" * 74)

        text = "\n".join(lines)
        print(text)
        with open(os.path.join(out_dir, 'model_complexity.txt'), 'w') as f:
            f.write(text)
        with open(os.path.join(out_dir, 'model_complexity.json'), 'w') as f:
            json.dump(report, f, indent=2)
        return report


# ==========================================
# 5. Trainer Class with Stabilizations
# ==========================================
class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, config):
        self.model = model.to(config.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.config = config
        
        # Replaced CosineAnnealing with OneCycleLR. 
        # Vision architectures require a strict linear warmup during the first 10% of training 
        # to prevent them from dropping into local minima (like predicting class 0 forever).
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, 
            max_lr=config.lr, 
            epochs=config.epochs, 
            steps_per_epoch=len(train_loader),
            pct_start=0.1 # 10% warmup phase
        )

        self.start_epoch = 0
        self.train_losses, self.val_losses = [], []
        self.val_accs_type, self.val_accs_stage = [], []
        self.best_val_loss = float('inf')

    def save_checkpoint(self, epoch, is_best=False):
        state = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'metrics': {
                'train_loss': self.train_losses, 'val_loss': self.val_losses,
                'val_acc_type': self.val_accs_type, 'val_acc_stage': self.val_accs_stage
            },
            'best_val_loss': self.best_val_loss
        }
        torch.save(state, self.config.checkpoint_path)
        if is_best:
            best_path = os.path.join(self.config.ckpt_dir, 'mtpg_net_best.pth')
            torch.save(state, best_path)

    def load_checkpoint(self):
        if os.path.exists(self.config.checkpoint_path):
            ckpt = torch.load(self.config.checkpoint_path, map_location=self.config.device)
            self.model.load_state_dict(ckpt['model_state_dict'])
            self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            if 'scheduler_state_dict' in ckpt:
                self.scheduler.load_state_dict(ckpt['scheduler_state_dict'])
            self.start_epoch = ckpt['epoch'] + 1
            self.train_losses = ckpt['metrics']['train_loss']
            self.val_losses = ckpt['metrics']['val_loss']
            self.val_accs_type = ckpt['metrics']['val_acc_type']
            self.val_accs_stage = ckpt['metrics']['val_acc_stage']
            self.best_val_loss = ckpt.get('best_val_loss', min(self.val_losses) if self.val_losses else float('inf'))
            print(f"[Trainer] Resumed automatically from epoch {self.start_epoch}")

    def plot_curves(self):
        metrics_dir = os.path.join(self.config.output_dir, 'metrics')
        os.makedirs(metrics_dir, exist_ok=True)

        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(self.train_losses, label='Train Loss', color='blue')
        plt.plot(self.val_losses, label='Val Loss', color='red')
        plt.title('Training vs Validation Loss')
        plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(self.val_accs_type, label='Level 1 (Type) Acc', color='green')
        plt.plot(self.val_accs_stage, label='Level 2 (Stage) Acc', color='purple')
        plt.title('Validation Accuracy')
        plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.legend()

        plt.tight_layout()
        plt.savefig(os.path.join(metrics_dir, 'train_valid_curves.png'), dpi=150)
        plt.close()

    def train(self):
        self.load_checkpoint()
        for epoch in range(self.start_epoch, self.config.epochs):
            self.model.train()
            epoch_loss = 0

            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.config.epochs}")
            for x, y_type, y_stage in pbar:
                x = x.to(self.config.device, non_blocking=True)
                y_type = y_type.to(self.config.device, non_blocking=True)
                y_stage = y_stage.to(self.config.device, non_blocking=True)

                self.optimizer.zero_grad(set_to_none=True)
                logits_type, logits_stage, _, _, _ = self.model(x)

                loss_type = self.criterion(logits_type, y_type)
                loss_stage = self.criterion(logits_stage, y_stage)
                loss = loss_type + loss_stage

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                
                self.optimizer.step()
                # OneCycleLR requires the scheduler to step PER BATCH, not per epoch.
                self.scheduler.step() 
                
                epoch_loss += loss.item()
                pbar.set_postfix({'loss': f"{loss.item():.4f}"})

            self.train_losses.append(epoch_loss / len(self.train_loader))
            val_loss_current = self.validate()

            is_best = val_loss_current < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss_current

            self.save_checkpoint(epoch, is_best=is_best)
            self.plot_curves()

    def validate(self):
        self.model.eval()
        val_loss, correct_type, correct_stage = 0, 0, 0
        n_seen = 0
        with torch.no_grad():
            for x, y_type, y_stage in self.val_loader:
                x = x.to(self.config.device, non_blocking=True)
                y_type = y_type.to(self.config.device, non_blocking=True)
                y_stage = y_stage.to(self.config.device, non_blocking=True)
                logits_type, logits_stage, _, _, _ = self.model(x)

                loss = self.criterion(logits_type, y_type) + self.criterion(logits_stage, y_stage)
                val_loss += loss.item()

                correct_type += (logits_type.argmax(dim=1) == y_type).sum().item()
                correct_stage += (logits_stage.argmax(dim=1) == y_stage).sum().item()
                n_seen += x.size(0)

        acc_type, acc_stage = correct_type / n_seen, correct_stage / n_seen
        avg_val_loss = val_loss / len(self.val_loader)

        self.val_losses.append(avg_val_loss)
        self.val_accs_type.append(acc_type)
        self.val_accs_stage.append(acc_stage)
        print(f"Val Loss: {avg_val_loss:.4f} | Acc Type: {acc_type:.4f} | Acc Stage: {acc_stage:.4f}")
        return avg_val_loss


# ==========================================
# 6. Inferencer Class (Comprehensive Metrics & Robust XAI)
# ==========================================
class Inferencer:
    def __init__(self, model, dataloader, type_names, stage_names, config):
        self.model = model.to(config.device)
        self.dataloader = dataloader
        self.type_names = type_names
        self.stage_names = stage_names
        self.config = config
        self.device = config.device

        self.dirs = {
            'metrics': os.path.join(config.output_dir, 'metrics'),
            'cm': os.path.join(config.output_dir, 'confusion_matrices'),
            'cm_per_cancer': os.path.join(config.output_dir, 'confusion_matrices', 'per_cancer'),
            'roc': os.path.join(config.output_dir, 'roc_curves'),
            'pr': os.path.join(config.output_dir, 'precision_recall_curves'),
            'xai': os.path.join(config.output_dir, 'xai_visualizations'),
            'bench': os.path.join(config.output_dir, 'benchmarks'),
            'complexity': os.path.join(config.output_dir, 'complexity'),
        }
        for d in self.dirs.values():
            os.makedirs(d, exist_ok=True)

        self.stage_to_type_idx = {}
        for si, sname in enumerate(self.stage_names):
            organ = sname.split('_')[0]
            if organ in self.type_names:
                self.stage_to_type_idx[si] = self.type_names.index(organ)

    def generate_all_explainability_artifacts(self):
        print("\n--- Generating Comprehensive Evaluation & XAI Artifacts ---")
        self.model.eval()

        analyzer = ComplexityAnalyzer(self.model, self.config)
        analyzer.run(self.dirs['complexity'])

        self.benchmark_pure_inference()
        self.plot_dataset_samples()

        all_logits_type, all_logits_stage = [], []
        all_labels_type, all_labels_stage = [], []
        all_feats = []
        sample_img, sample_type_label, sample_stage_label = None, None, None
        sample_attns, sample_routes = None, None

        for x, y_t, y_s in self.dataloader:
            sample_img = x[0:1].clone()
            sample_type_label = y_t[0:1].clone()
            sample_stage_label = y_s[0:1].clone()
            break

        with torch.no_grad():
            for i, (x, y_t, y_s) in enumerate(tqdm(self.dataloader, desc="Running Full Evaluation Pass")):
                x_dev = x.to(self.device, non_blocking=True)
                want_internals = (i == 0)
                logits_type, logits_stage, attns, routes, feats = self.model(
                    x_dev, return_internals=want_internals)

                all_logits_type.append(logits_type.detach().cpu())
                all_logits_stage.append(logits_stage.detach().cpu())
                all_feats.append(feats.detach().cpu())
                all_labels_type.append(y_t)
                all_labels_stage.append(y_s)

                if want_internals:
                    sample_attns = [a.detach().cpu() for a in attns]
                    sample_routes = [r.detach().cpu() for r in routes]

        logits_type = torch.cat(all_logits_type)
        logits_stage = torch.cat(all_logits_stage)
        feats_all = torch.cat(all_feats).numpy()
        y_true_type = torch.cat(all_labels_type).numpy()
        y_true_stage = torch.cat(all_labels_stage).numpy()

        probs_type = F.softmax(logits_type, dim=-1).numpy()
        probs_stage = F.softmax(logits_stage, dim=-1).numpy()
        y_pred_type = probs_type.argmax(axis=1)
        y_pred_stage = probs_stage.argmax(axis=1)

        self.calculate_and_print_metrics(y_true_type, y_pred_type, probs_type, self.type_names, "Level_1_Type")
        self.calculate_and_print_metrics(y_true_stage, y_pred_stage, probs_stage, self.stage_names, "Level_2_Stage")
        self.hierarchical_consistency_report(y_true_type, y_pred_type, y_true_stage, y_pred_stage)

        self.plot_confusion_matrix(y_true_type, y_pred_type, self.type_names, "Level_1_Type")
        self.plot_confusion_matrix(y_true_stage, y_pred_stage, self.stage_names, "Level_2_Stage")
        self.plot_per_cancer_confusion_matrices(y_true_type, y_pred_type, y_true_stage, y_pred_stage)

        self.plot_tsne(feats_all, y_true_stage, self.stage_names)
        self.plot_roc_curves(y_true_type, probs_type, self.type_names, "Level_1_Type")
        self.plot_roc_curves(y_true_stage, probs_stage, self.stage_names, "Level_2_Stage")
        self.plot_pr_curves(y_true_type, probs_type, self.type_names, "Level_1_Type")
        self.plot_pr_curves(y_true_stage, probs_stage, self.stage_names, "Level_2_Stage")
        self.plot_calibration(y_true_stage, probs_stage, "Level_2_Stage")

        if sample_attns is not None:
            self.plot_novel_attention(sample_attns)
            self.plot_attention_rollout(sample_attns, sample_img)
        if sample_routes is not None:
            self.plot_morphology_routes(sample_routes)

        tgt = int(sample_stage_label.item())
        print("[XAI] Grad-CAM ...");            self.explain_gradcam(sample_img, tgt)
        print("[XAI] Grad-CAM++ ...");          self.explain_gradcam_plusplus(sample_img, tgt)
        print("[XAI] Score-CAM ...");           self.explain_scorecam(sample_img, tgt)
        print("[XAI] Ablation-CAM ...");        self.explain_ablationcam(sample_img, tgt)
        print("[XAI] Vanilla Saliency ...");    self.explain_saliency(sample_img, tgt)
        print("[XAI] Guided Backprop ...");     self.explain_guided_backprop(sample_img, tgt)
        print("[XAI] Integrated Gradients ...");self.explain_integrated_gradients(sample_img, tgt)
        print("[XAI] Occlusion (Glimpse) ..."); self.explain_occlusion(sample_img, tgt)
        print("[XAI] LIME ...");                self.explain_lime(sample_img, tgt)
        print("[XAI] SHAP ...");                self.explain_shap(sample_img, tgt)
        print("[XAI] XAI consensus panel ..."); self.explain_consensus_panel(sample_img, tgt)

        print("All Evaluation and XAI Visualizations generated successfully!")

    def _to_display(self, img_tensor):
        arr = img_tensor.detach().squeeze(0).cpu().permute(1, 2, 0).numpy()
        return np.clip(arr, 0, 1)

    def _overlay(self, base_hwc, heat_2d, alpha=0.45, cmap=cv2.COLORMAP_JET):
        heat = np.clip(heat_2d, 0, 1)
        heat_u8 = np.uint8(255 * heat)
        colored = cv2.applyColorMap(heat_u8, cmap)
        colored = cv2.cvtColor(colored, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        out = (1 - alpha) * base_hwc + alpha * colored
        return np.clip(out, 0, 1)

    @staticmethod
    def _norm01(a):
        a = a - a.min()
        d = a.max()
        return a / (d + 1e-8)

    def _batched_stage_probs(self, batch_tensor, chunk=64):
        outs = []
        with torch.no_grad():
            for i in range(0, batch_tensor.shape[0], chunk):
                sub = batch_tensor[i:i + chunk].to(self.device)
                _, ls, _, _, _ = self.model(sub)
                outs.append(F.softmax(ls, dim=-1).cpu())
        return torch.cat(outs).numpy()

    def benchmark_pure_inference(self):
        cfg = self.config
        self.model.eval()
        results = {}

        lines = ["=" * 74, "PURE MODEL INFERENCE BENCHMARK (forward pass only)", "=" * 74,
                 f"Device: {self.device}"]
        if self.device == 'cuda':
            lines.append(f"GPU: {torch.cuda.get_device_name(0)}")
        lines.append(f"Warmup iters: {cfg.bench_warmup_iters}  Measured iters: {cfg.bench_measure_iters}")
        lines.append("")

        for bs in cfg.bench_batch_sizes:
            dummy = torch.randn(bs, cfg.in_chans, cfg.img_size, cfg.img_size,
                                device=self.device)

            with torch.no_grad():
                for _ in range(cfg.bench_warmup_iters):
                    self.model(dummy)
                if self.device == 'cuda':
                    torch.cuda.synchronize()

                timings = []
                for _ in range(cfg.bench_measure_iters):
                    if self.device == 'cuda':
                        torch.cuda.synchronize()
                    t0 = time.perf_counter()
                    self.model(dummy)
                    if self.device == 'cuda':
                        torch.cuda.synchronize()
                    timings.append((time.perf_counter() - t0) * 1000.0)

            t = np.array(timings)
            entry = {
                'batch_size': bs,
                'mean_batch_latency_ms': float(t.mean()),
                'std_batch_latency_ms': float(t.std()),
                'median_batch_latency_ms': float(np.median(t)),
                'p90_batch_latency_ms': float(np.percentile(t, 90)),
                'p95_batch_latency_ms': float(np.percentile(t, 95)),
                'p99_batch_latency_ms': float(np.percentile(t, 99)),
                'min_batch_latency_ms': float(t.min()),
                'max_batch_latency_ms': float(t.max()),
                'mean_per_image_ms': float(t.mean() / bs),
                'throughput_images_per_sec': float(bs / (t.mean() / 1000.0)),
            }
            results[f'batch_{bs}'] = entry

            lines.append(f"Batch size {bs}")
            lines.append(f"  Mean batch latency    : {entry['mean_batch_latency_ms']:.4f} ms "
                         f"(+/- {entry['std_batch_latency_ms']:.4f})")
            lines.append(f"  Median / p90 / p99    : {entry['median_batch_latency_ms']:.4f} / "
                         f"{entry['p90_batch_latency_ms']:.4f} / {entry['p99_batch_latency_ms']:.4f} ms")
            lines.append(f"  PURE per-image time   : {entry['mean_per_image_ms']:.4f} ms")
            lines.append(f"  Throughput            : {entry['throughput_images_per_sec']:.2f} img/s")
            lines.append("")

            del dummy
            if self.device == 'cuda':
                torch.cuda.empty_cache()

        if self.device == 'cuda':
            torch.cuda.reset_peak_memory_stats()
            d = torch.randn(1, cfg.in_chans, cfg.img_size, cfg.img_size, device=self.device)
            with torch.no_grad():
                self.model(d)
            torch.cuda.synchronize()
            peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
            results['peak_gpu_memory_MB_bs1'] = float(peak)
            lines.append(f"Peak GPU memory (bs=1): {peak:.2f} MB")
            del d
            torch.cuda.empty_cache()

        lines.append("=" * 74)
        text = "\n".join(lines)
        print(text)
        with open(os.path.join(self.dirs['bench'], 'pure_inference_benchmark.txt'), 'w') as f:
            f.write(text)
        with open(os.path.join(self.dirs['bench'], 'pure_inference_benchmark.json'), 'w') as f:
            json.dump(results, f, indent=2)

        bss = [r['batch_size'] for k, r in results.items() if k.startswith('batch_')]
        pim = [r['mean_per_image_ms'] for k, r in results.items() if k.startswith('batch_')]
        thr = [r['throughput_images_per_sec'] for k, r in results.items() if k.startswith('batch_')]
        fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
        ax[0].plot(bss, pim, 'o-'); ax[0].set_xlabel('Batch size')
        ax[0].set_ylabel('Pure per-image latency (ms)'); ax[0].set_title('Latency'); ax[0].grid(alpha=.3)
        ax[1].plot(bss, thr, 's-', color='darkgreen'); ax[1].set_xlabel('Batch size')
        ax[1].set_ylabel('Images / sec'); ax[1].set_title('Throughput'); ax[1].grid(alpha=.3)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['bench'], 'inference_scaling.png'), dpi=150)
        plt.close()
        return results

    def calculate_and_print_metrics(self, y_true, y_pred, probs, target_names, task_name):
        acc = accuracy_score(y_true, y_pred)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        prec_w = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec_w = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1_w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(y_true, y_pred)
        mcc = matthews_corrcoef(y_true, y_pred)

        present = np.unique(y_true)
        y_true_bin = label_binarize(y_true, classes=range(len(target_names)))
        if len(target_names) == 2:
            y_true_bin = np.hstack((1 - y_true_bin, y_true_bin))

        try:
            roc_auc = roc_auc_score(y_true_bin[:, present], probs[:, present],
                                    average='macro', multi_class='ovr')
        except Exception:
            roc_auc = float('nan')
        try:
            pr_auc = average_precision_score(y_true_bin[:, present], probs[:, present], average='macro')
        except Exception:
            pr_auc = float('nan')

        rmse = float(np.sqrt(np.mean((probs - y_true_bin) ** 2)))
        try:
            top5 = top_k_accuracy_score(y_true, probs, k=min(5, probs.shape[1]),
                                        labels=list(range(len(target_names))))
        except Exception:
            top5 = float('nan')

        cm = confusion_matrix(y_true, y_pred, labels=range(len(target_names)))
        specificities = []
        for i in range(len(target_names)):
            tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
            fp = np.sum(cm[:, i]) - cm[i, i]
            specificities.append(tn / (tn + fp + 1e-8))
        macro_spec = float(np.mean(specificities))

        report = classification_report(y_true, y_pred,
                                       labels=range(len(target_names)),
                                       target_names=target_names, zero_division=0)

        per_class_lines = ["\nPer-class Specificity:"]
        for n, s in zip(target_names, specificities):
            per_class_lines.append(f"  {n:<34s} {s:.4f}")

        output_txt = (
            f"=== {task_name} Metrics ===\n"
            f"Accuracy            : {acc:.4f}\n"
            f"Balanced Accuracy   : {bal_acc:.4f}\n"
            f"Top-5 Accuracy      : {top5:.4f}\n"
            f"Precision (macro)   : {prec:.4f}\n"
            f"Recall/Sens (macro) : {rec:.4f}\n"
            f"Specificity (macro) : {macro_spec:.4f}\n"
            f"F1 (macro)          : {f1:.4f}\n"
            f"Precision (weighted): {prec_w:.4f}\n"
            f"Recall (weighted)   : {rec_w:.4f}\n"
            f"F1 (weighted)       : {f1_w:.4f}\n"
            f"Cohen's Kappa       : {kappa:.4f}\n"
            f"MCC                 : {mcc:.4f}\n"
            f"ROC-AUC (macro OvR) : {roc_auc:.4f}\n"
            f"PR-AUC (macro)      : {pr_auc:.4f}\n"
            f"RMSE                : {rmse:.4f}\n"
            + "\n".join(per_class_lines) +
            f"\n\nClassification Report:\n{report}\n"
        )
        print(output_txt)
        with open(os.path.join(self.dirs['metrics'], f"{task_name}_metrics.txt"), 'w') as f:
            f.write(output_txt)

    def hierarchical_consistency_report(self, yt_true, yt_pred, ys_true, ys_pred):
        implied = np.array([self.stage_to_type_idx.get(int(s), -1) for s in ys_pred])
        valid = implied >= 0
        consistency = float((implied[valid] == yt_pred[valid]).mean())

        implied_true = np.array([self.stage_to_type_idx.get(int(s), -1) for s in ys_true])
        organ_from_stage_acc = float((implied[valid] == implied_true[valid]).mean())

        txt = (
            "=== Hierarchical Consistency ===\n"
            f"Type-head vs Stage-head agreement : {consistency:.4f}\n"
            f"Organ accuracy implied by stage   : {organ_from_stage_acc:.4f}\n"
            f"Direct organ-head accuracy        : {float((yt_pred == yt_true).mean()):.4f}\n"
        )
        print(txt)
        with open(os.path.join(self.dirs['metrics'], 'hierarchical_consistency.txt'), 'w') as f:
            f.write(txt)

    def plot_dataset_samples(self):
        plt.figure(figsize=(10, 10))
        count = 0
        for x, y_t, y_s in self.dataloader:
            for i in range(min(9 - count, x.shape[0])):
                img = np.clip(x[i].permute(1, 2, 0).numpy(), 0, 1)
                plt.subplot(3, 3, count + 1)
                plt.imshow(img)
                plt.title(f"{self.type_names[y_t[i].item()]}\n"
                          f"({self.stage_names[y_s[i].item()].split('_', 1)[-1]})", fontsize=9)
                plt.axis('off')
                count += 1
            if count >= 9:
                break
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['metrics'], 'dataset_samples.png'), dpi=150)
        plt.close()

    def plot_confusion_matrix(self, y_true, y_pred, labels, title, out_dir=None, normalize=True):
        out_dir = out_dir or self.dirs['cm']
        cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))

        fig, axes = plt.subplots(1, 2, figsize=(2 + 2.2 * len(labels), 1 + 1.0 * len(labels)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=False)
        axes[0].set_title(f'{title} — Counts')
        axes[0].set_ylabel('True Class'); axes[0].set_xlabel('Predicted Class')

        if normalize:
            with np.errstate(all='ignore'):
                cmn = cm.astype(np.float64) / (cm.sum(axis=1, keepdims=True) + 1e-12)
            sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
                        xticklabels=labels, yticklabels=labels, ax=axes[1], cbar=True)
            axes[1].set_title(f'{title} — Row-normalized (recall)')
            axes[1].set_ylabel('True Class'); axes[1].set_xlabel('Predicted Class')

        for ax in axes:
            ax.tick_params(axis='x', rotation=45)
            for lbl in ax.get_xticklabels():
                lbl.set_ha('right')
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f'{title}_confusion_matrix.png'), dpi=150)
        plt.close()
        return cm

    def plot_per_cancer_confusion_matrices(self, yt_true, yt_pred, ys_true, ys_pred):
        summary = []
        for ti, tname in enumerate(self.type_names):
            local_stage_ids = [si for si, s in enumerate(self.stage_names)
                               if self.stage_to_type_idx.get(si, -1) == ti]
            if len(local_stage_ids) == 0:
                continue

            mask = (ys_true == np.array(local_stage_ids)[:, None]).any(axis=0)
            if mask.sum() == 0:
                continue

            local_map = {gid: k for k, gid in enumerate(local_stage_ids)}
            K = len(local_stage_ids)
            OUT = K 

            t_local = np.array([local_map[int(s)] for s in ys_true[mask]])
            p_local = np.array([local_map.get(int(s), OUT) for s in ys_pred[mask]])

            labels_local = [self.stage_names[g].split('_', 1)[-1] for g in local_stage_ids]
            cm = np.zeros((K, K + 1), dtype=np.int64)
            for a, b in zip(t_local, p_local):
                cm[a, b] += 1

            col_labels = labels_local + ['OUT_OF_ORGAN']
            fig, axes = plt.subplots(1, 2, figsize=(4 + 1.9 * (K + 1), 2 + 1.0 * K))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                        xticklabels=col_labels, yticklabels=labels_local, ax=axes[0], cbar=False)
            axes[0].set_title(f'{tname} — Counts (n={int(mask.sum())})')
            axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

            with np.errstate(all='ignore'):
                cmn = cm.astype(np.float64) / (cm.sum(axis=1, keepdims=True) + 1e-12)
            sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Purples', vmin=0, vmax=1,
                        xticklabels=col_labels, yticklabels=labels_local, ax=axes[1], cbar=True)
            axes[1].set_title(f'{tname} — Row-normalized')
            axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

            for ax in axes:
                ax.tick_params(axis='x', rotation=45)
                for lbl in ax.get_xticklabels():
                    lbl.set_ha('right')
            plt.tight_layout()
            safe = tname.replace(' ', '_')
            plt.savefig(os.path.join(self.dirs['cm_per_cancer'],
                                     f'{safe}_stage_confusion_matrix.png'), dpi=150)
            plt.close()

            in_organ = cm[:, :K]
            acc = float(np.trace(in_organ) / max(cm.sum(), 1))
            leak = float(cm[:, OUT].sum() / max(cm.sum(), 1))
            summary.append((tname, int(mask.sum()), K, acc, leak))

        for ti, tname in enumerate(self.type_names):
            mask = (yt_true == ti)
            if mask.sum() == 0:
                continue
            bt = np.ones(int(mask.sum()), dtype=int)
            bp = (yt_pred[mask] == ti).astype(int)
            cm = confusion_matrix(bt, bp, labels=[0, 1])
            plt.figure(figsize=(4.2, 3.6))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                        xticklabels=[f'not {tname}', tname],
                        yticklabels=[f'not {tname}', tname], cbar=False)
            plt.title(f'{tname} — organ detection')
            plt.tight_layout()
            safe = tname.replace(' ', '_')
            plt.savefig(os.path.join(self.dirs['cm_per_cancer'],
                                     f'{safe}_organ_detection_cm.png'), dpi=150)
            plt.close()

        lines = ["=== Per-Cancer Confusion Matrix Summary ===",
                 f"{'Organ':<14s}{'N':>7s}{'Classes':>9s}{'StageAcc':>11s}{'OrganLeak':>11s}"]
        for tname, n, k, acc, leak in summary:
            lines.append(f"{tname:<14s}{n:>7d}{k:>9d}{acc:>11.4f}{leak:>11.4f}")
        txt = "\n".join(lines)
        print("\n" + txt)
        with open(os.path.join(self.dirs['cm_per_cancer'], 'per_cancer_summary.txt'), 'w') as f:
            f.write(txt)

    def plot_roc_curves(self, y_true, probs, classes, title):
        y_true_bin = label_binarize(y_true, classes=range(len(classes)))
        if len(classes) == 2:
            y_true_bin = np.hstack((1 - y_true_bin, y_true_bin))
        plt.figure(figsize=(11, 8))
        for i, cls_name in enumerate(classes):
            if y_true_bin[:, i].sum() == 0:
                continue
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], probs[:, i])
            plt.plot(fpr, tpr, lw=1.8, label=f'{cls_name} (AUC={auc(fpr, tpr):.3f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=1.2)
        plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curves: {title}')
        plt.legend(loc="lower right", fontsize=8)
        plt.grid(alpha=.3)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['roc'], f'{title}_roc_curve.png'), dpi=150)
        plt.close()

    def plot_pr_curves(self, y_true, probs, classes, title):
        y_true_bin = label_binarize(y_true, classes=range(len(classes)))
        if len(classes) == 2:
            y_true_bin = np.hstack((1 - y_true_bin, y_true_bin))
        plt.figure(figsize=(11, 8))
        for i, cls_name in enumerate(classes):
            if y_true_bin[:, i].sum() == 0:
                continue
            prec, rec, _ = precision_recall_curve(y_true_bin[:, i], probs[:, i])
            ap = average_precision_score(y_true_bin[:, i], probs[:, i])
            plt.plot(rec, prec, lw=1.8, label=f'{cls_name} (AP={ap:.3f})')
        plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
        plt.xlabel('Recall'); plt.ylabel('Precision')
        plt.title(f'Precision-Recall Curves: {title}')
        plt.legend(loc="lower left", fontsize=8)
        plt.grid(alpha=.3)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['pr'], f'{title}_pr_curve.png'), dpi=150)
        plt.close()

    def plot_calibration(self, y_true, probs, title):
        conf = probs.max(axis=1)
        correct = (probs.argmax(axis=1) == y_true).astype(int)
        try:
            frac_pos, mean_pred = calibration_curve(correct, conf, n_bins=15, strategy='quantile')
        except Exception:
            return
        bins = np.linspace(0, 1, 16)
        idx = np.digitize(conf, bins) - 1
        ece = 0.0
        for b in range(15):
            m = idx == b
            if m.sum() == 0:
                continue
            ece += (m.sum() / len(conf)) * abs(correct[m].mean() - conf[m].mean())

        plt.figure(figsize=(6.5, 6))
        plt.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
        plt.plot(mean_pred, frac_pos, 'o-', label=f'CanceRX (ECE={ece:.4f})')
        plt.xlabel('Mean predicted confidence'); plt.ylabel('Empirical accuracy')
        plt.title(f'Reliability Diagram: {title}')
        plt.legend(); plt.grid(alpha=.3)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['metrics'], f'{title}_calibration.png'), dpi=150)
        plt.close()

    def plot_tsne(self, features, labels, class_names):
        n = features.shape[0]
        if n > 5000:
            sel = np.random.RandomState(0).choice(n, 5000, replace=False)
            features, labels = features[sel], labels[sel]
        perp = max(5, min(30, features.shape[0] // 4))
        tsne = TSNE(n_components=2, perplexity=perp, random_state=42,
                    init='pca', learning_rate='auto').fit_transform(features)
        plt.figure(figsize=(11, 9))
        scatter = plt.scatter(tsne[:, 0], tsne[:, 1], c=labels, cmap='tab20', alpha=0.75, s=8)
        unique_labels = np.unique(labels)
        
        handles = [mpatches.Patch(color=scatter.cmap(scatter.norm(c)), label=class_names[c])
                   for c in unique_labels]
                   
        plt.legend(handles=handles, title="Stages", bbox_to_anchor=(1.02, 1),
                   loc='upper left', fontsize=8)
        plt.title('t-SNE of Penultimate Feature Space')
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['metrics'], 'tsne_projection.png'), dpi=150)
        plt.close()

    def plot_novel_attention(self, attns):
        g = self.model.grid
        n_layers = len(attns)
        fig, axes = plt.subplots(2, n_layers, figsize=(4.0 * n_layers, 8))
        if n_layers == 1:
            axes = axes.reshape(2, 1)

        for li, a in enumerate(attns):
            A = a[0] 
            token_importance = A.mean(dim=0).mean(dim=0)
            grid_map = token_importance.reshape(g, g).numpy()
            grid_map = self._norm01(grid_map)

            im0 = axes[0, li].imshow(grid_map, cmap='magma')
            axes[0, li].set_title(f'Layer {li+1}: PGEA token salience')
            axes[0, li].axis('off')
            plt.colorbar(im0, ax=axes[0, li], fraction=0.046)

            im1 = axes[1, li].imshow(A.mean(dim=0).numpy(), cmap='viridis')
            axes[1, li].set_title(f'Layer {li+1}: full {A.shape[-1]}x{A.shape[-1]} map')
            axes[1, li].axis('off')
            plt.colorbar(im1, ax=axes[1, li], fraction=0.046)

        plt.suptitle('Phase-Gated Entanglement Attention', fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'pgea_attention.png'), dpi=150)
        plt.close()

    def plot_attention_rollout(self, attns, img_tensor):
        g = self.model.grid
        N = g * g
        result = torch.eye(N)
        for a in attns:
            A = a[0].mean(dim=0)
            A = A + torch.eye(N)
            A = A / A.sum(dim=-1, keepdim=True)
            result = A @ result
        roll = result.mean(dim=0).reshape(g, g).numpy()
        roll = self._norm01(roll)
        roll_full = cv2.resize(roll, (self.config.img_size, self.config.img_size),
                               interpolation=cv2.INTER_CUBIC)
        base = self._to_display(img_tensor)

        fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
        ax[0].imshow(base); ax[0].set_title('Input'); ax[0].axis('off')
        ax[1].imshow(roll_full, cmap='inferno'); ax[1].set_title('Attention Rollout'); ax[1].axis('off')
        ax[2].imshow(self._overlay(base, roll_full)); ax[2].set_title('Overlay'); ax[2].axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'attention_rollout.png'), dpi=150)
        plt.close()

    def plot_morphology_routes(self, routes):
        n_layers = len(routes)
        fig, axes = plt.subplots(2, n_layers, figsize=(4.2 * n_layers, 7.5))
        if n_layers == 1:
            axes = axes.reshape(2, 1)
        g = self.model.grid
        for li, r in enumerate(routes):
            R = r[0].numpy()
            sns.heatmap(R.T, cmap='viridis', ax=axes[0, li], cbar=True)
            axes[0, li].set_title(f'Layer {li+1}: expert weights per token')
            axes[0, li].set_xlabel('Token'); axes[0, li].set_ylabel('Expert')

            dominant = R.argmax(axis=1).reshape(g, g)
            im = axes[1, li].imshow(dominant, cmap='tab10',
                                    vmin=0, vmax=max(1, R.shape[1] - 1))
            axes[1, li].set_title(f'Layer {li+1}: dominant expert (spatial)')
            axes[1, li].axis('off')
            plt.colorbar(im, ax=axes[1, li], fraction=0.046)
        plt.suptitle('Topological Pathway Routing (dense MoE)', fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'tpr_routes.png'), dpi=150)
        plt.close()

    def _cam_forward_backward(self, img_tensor, target_class):
        self.model.eval()
        self.model.enable_cam_hooks()
        x = img_tensor.clone().to(self.device)
        x.requires_grad_(True)

        with torch.enable_grad():
            _, logits_stage, _, _, _ = self.model(x)
            score = logits_stage[0, target_class]
            self.model.zero_grad(set_to_none=True)
            score.backward()

        acts = self.model.cam_activations.detach()[0]
        grads = self.model.cam_gradients.detach()[0]
        self.model.disable_cam_hooks()
        return acts, grads

    def explain_gradcam(self, img_tensor, target_stage_class):
        acts, grads = self._cam_forward_backward(img_tensor, target_stage_class)
        weights = grads.mean(dim=(1, 2))
        cam = torch.relu((weights[:, None, None] * acts).sum(0)).cpu().numpy()
        cam = self._norm01(cam)
        cam_full = cv2.resize(cam, (self.config.img_size, self.config.img_size),
                              interpolation=cv2.INTER_CUBIC)
        self._save_cam_figure(img_tensor, cam_full, target_stage_class,
                              'Grad-CAM', 'gradcam_result.png')
        return cam_full

    def explain_gradcam_plusplus(self, img_tensor, target_stage_class):
        acts, grads = self._cam_forward_backward(img_tensor, target_stage_class)
        g2 = grads ** 2
        g3 = grads ** 3
        sum_a = acts.sum(dim=(1, 2))[:, None, None]
        denom = 2.0 * g2 + sum_a * g3
        denom = torch.where(denom != 0, denom, torch.ones_like(denom))
        alpha = g2 / denom
        weights = (alpha * torch.relu(grads)).sum(dim=(1, 2))
        cam = torch.relu((weights[:, None, None] * acts).sum(0)).cpu().numpy()
        cam = self._norm01(cam)
        cam_full = cv2.resize(cam, (self.config.img_size, self.config.img_size),
                              interpolation=cv2.INTER_CUBIC)
        self._save_cam_figure(img_tensor, cam_full, target_stage_class,
                              'Grad-CAM++', 'gradcam_plusplus_result.png')
        return cam_full

    def explain_scorecam(self, img_tensor, target_stage_class):
        self.model.eval()
        self.model.enable_cam_hooks()
        x = img_tensor.clone().to(self.device)
        with torch.no_grad():
            _, base_logits, _, _, _ = self.model(x)
            base_prob = F.softmax(base_logits, dim=-1)[0, target_stage_class].item()
        acts = self.model.cam_activations.detach()[0]
        self.model.disable_cam_hooks()

        energy = acts.flatten(1).max(dim=1).values
        topk = min(self.config.scorecam_topk, acts.shape[0])
        idx = torch.topk(energy, topk).indices

        masks = []
        for c in idx:
            m = acts[c]
            m = (m - m.min()) / (m.max() - m.min() + 1e-8)
            m = F.interpolate(m[None, None], size=(self.config.img_size, self.config.img_size),
                              mode='bicubic', align_corners=False)[0, 0]
            masks.append(m)
        masks = torch.stack(masks)

        scores = []
        with torch.no_grad():
            for i in range(0, topk, 16):
                chunk = masks[i:i + 16]
                batch = x.repeat(chunk.shape[0], 1, 1, 1) * chunk[:, None]
                _, ls, _, _, _ = self.model(batch)
                scores.append(F.softmax(ls, dim=-1)[:, target_stage_class].cpu())
        scores = torch.cat(scores)
        w = F.softmax(scores, dim=0)

        cam = (w[:, None, None] * masks.cpu()).sum(0).numpy()
        cam = self._norm01(np.maximum(cam, 0))
        self._save_cam_figure(img_tensor, cam, target_stage_class,
                              f'Score-CAM (base p={base_prob:.3f})', 'scorecam_result.png')
        return cam

    def explain_ablationcam(self, img_tensor, target_stage_class):
        self.model.eval()
        self.model.enable_cam_hooks()
        x = img_tensor.clone().to(self.device)
        with torch.no_grad():
            _, base_logits, _, _, _ = self.model(x)
        base_score = base_logits[0, target_stage_class].item()
        acts = self.model.cam_activations.detach()[0].clone()
        self.model.disable_cam_hooks()

        D, g, _ = acts.shape
        energy = acts.flatten(1).abs().mean(dim=1)
        topk = min(self.config.scorecam_topk, D)
        idx = torch.topk(energy, topk).indices

        weights = torch.zeros(D, device=acts.device)
        stash = {}

        def ablate_hook(module, inp, out):
            o = out.clone()
            o[:, stash['c']] = 0
            return o

        for c in idx:
            stash['c'] = int(c)
            h = self.model.patch_embed.register_forward_hook(ablate_hook)
            with torch.no_grad():
                _, ls, _, _, _ = self.model(x)
            h.remove()
            weights[c] = (base_score - ls[0, target_stage_class].item()) / (abs(base_score) + 1e-8)

        cam = torch.relu((weights[:, None, None] * acts).sum(0)).cpu().numpy()
        cam = self._norm01(cam)
        cam_full = cv2.resize(cam, (self.config.img_size, self.config.img_size),
                              interpolation=cv2.INTER_CUBIC)
        self._save_cam_figure(img_tensor, cam_full, target_stage_class,
                              'Ablation-CAM', 'ablationcam_result.png')
        return cam_full

    def _save_cam_figure(self, img_tensor, cam_full, target_class, method, fname):
        base = self._to_display(img_tensor)
        fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
        ax[0].imshow(base); ax[0].set_title("Input (preprocessed)"); ax[0].axis('off')
        im = ax[1].imshow(cam_full, cmap='jet'); ax[1].set_title(f"{method} map"); ax[1].axis('off')
        plt.colorbar(im, ax=ax[1], fraction=0.046)
        ax[2].imshow(self._overlay(base, cam_full))
        ax[2].set_title(f"Target: {self.stage_names[target_class]}", fontsize=9); ax[2].axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], fname), dpi=150)
        plt.close()

    def explain_saliency(self, img_tensor, target_stage_class):
        self.model.eval()
        x = img_tensor.clone().to(self.device).requires_grad_(True)
        with torch.enable_grad():
            _, ls, _, _, _ = self.model(x)
            self.model.zero_grad(set_to_none=True)
            ls[0, target_stage_class].backward()
        sal = x.grad.detach()[0].abs().max(dim=0).values.cpu().numpy()
        sal = self._norm01(sal)
        base = self._to_display(img_tensor)

        fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
        ax[0].imshow(base); ax[0].set_title('Input'); ax[0].axis('off')
        ax[1].imshow(sal, cmap='hot'); ax[1].set_title('Vanilla Saliency'); ax[1].axis('off')
        ax[2].imshow(self._overlay(base, sal, alpha=0.5)); ax[2].set_title('Overlay'); ax[2].axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'saliency_result.png'), dpi=150)
        plt.close()
        return sal

    def explain_guided_backprop(self, img_tensor, target_stage_class):
        self.model.eval()
        handles = []

        def guided_hook(module, grad_in, grad_out):
            return (torch.clamp(grad_in[0], min=0.0),)

        for m in self.model.modules():
            if isinstance(m, (nn.GELU, nn.ReLU)):
                handles.append(m.register_full_backward_hook(guided_hook))

        x = img_tensor.clone().to(self.device).requires_grad_(True)
        with torch.enable_grad():
            _, ls, _, _, _ = self.model(x)
            self.model.zero_grad(set_to_none=True)
            ls[0, target_stage_class].backward()
        gb = x.grad.detach()[0].cpu().numpy()

        for h in handles:
            h.remove()

        gb_vis = np.transpose(gb, (1, 2, 0))
        gb_vis = self._norm01(gb_vis)
        gb_gray = self._norm01(np.abs(gb).max(axis=0))
        base = self._to_display(img_tensor)

        fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
        ax[0].imshow(base); ax[0].set_title('Input'); ax[0].axis('off')
        ax[1].imshow(gb_vis); ax[1].set_title('Guided Backprop (RGB)'); ax[1].axis('off')
        ax[2].imshow(gb_gray, cmap='gray'); ax[2].set_title('Guided Backprop (magnitude)'); ax[2].axis('off')
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'guided_backprop_result.png'), dpi=150)
        plt.close()
        return gb_gray

    def explain_integrated_gradients(self, img_tensor, target_stage_class):
        self.model.eval()
        x = img_tensor.clone().to(self.device)
        baseline = torch.zeros_like(x)
        steps = self.config.ig_steps

        total_grad = torch.zeros_like(x)
        for i in range(steps):
            a = float(i + 1) / steps
            interp = (baseline + a * (x - baseline)).requires_grad_(True)
            with torch.enable_grad():
                _, ls, _, _, _ = self.model(interp)
                self.model.zero_grad(set_to_none=True)
                ls[0, target_stage_class].backward()
            total_grad += interp.grad.detach()

        avg_grad = total_grad / steps
        ig = ((x - baseline) * avg_grad).detach()[0].cpu().numpy()

        with torch.no_grad():
            _, l_x, _, _, _ = self.model(x)
            _, l_b, _, _, _ = self.model(baseline)
        delta = (l_x[0, target_stage_class] - l_b[0, target_stage_class]).item()
        completeness_err = abs(delta - ig.sum())

        ig_map = self._norm01(np.abs(ig).sum(axis=0))
        ig_signed = ig.sum(axis=0)
        vmax = np.abs(ig_signed).max() + 1e-8
        base = self._to_display(img_tensor)

        fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
        ax[0].imshow(base); ax[0].set_title('Input'); ax[0].axis('off')
        ax[1].imshow(ig_map, cmap='hot'); ax[1].set_title('|IG| attribution'); ax[1].axis('off')
        im = ax[2].imshow(ig_signed, cmap='coolwarm', vmin=-vmax, vmax=vmax)
        ax[2].set_title('Signed IG'); ax[2].axis('off'); plt.colorbar(im, ax=ax[2], fraction=0.046)
        ax[3].imshow(self._overlay(base, ig_map, alpha=0.5)); ax[3].set_title('Overlay'); ax[3].axis('off')
        plt.suptitle(f'Integrated Gradients — completeness error {completeness_err:.4f}', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'integrated_gradients_result.png'), dpi=150)
        plt.close()
        return ig_map

    def explain_occlusion(self, img_tensor, target_class):
        patch_size = self.config.occlusion_patch
        stride = self.config.occlusion_stride
        x = img_tensor.to(self.device)
        _, _, H, W = x.shape

        fill = x.mean(dim=(2, 3), keepdim=True)

        with torch.no_grad():
            _, orig_logit, _, _, _ = self.model(x)
            orig_prob = F.softmax(orig_logit, dim=-1)[0, target_class].item()

        positions = []
        for h in range(0, H - patch_size + 1, stride):
            for w in range(0, W - patch_size + 1, stride):
                positions.append((h, w))

        out_h = len(range(0, H - patch_size + 1, stride))
        out_w = len(range(0, W - patch_size + 1, stride))
        heat = np.zeros(len(positions), dtype=np.float32)

        BS = 32
        with torch.no_grad():
            for i in range(0, len(positions), BS):
                chunk = positions[i:i + BS]
                batch = x.repeat(len(chunk), 1, 1, 1)
                for j, (h, w) in enumerate(chunk):
                    batch[j, :, h:h + patch_size, w:w + patch_size] = fill[0]
                _, ls, _, _, _ = self.model(batch)
                p = F.softmax(ls, dim=-1)[:, target_class].cpu().numpy()
                heat[i:i + len(chunk)] = orig_prob - p

        heat = heat.reshape(out_h, out_w)
        heat_norm = self._norm01(np.maximum(heat, 0))
        heat_full = cv2.resize(heat_norm, (W, H), interpolation=cv2.INTER_CUBIC)
        base = self._to_display(img_tensor)

        fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
        ax[0].imshow(base); ax[0].set_title("Input"); ax[0].axis('off')
        im = ax[1].imshow(heat, cmap='viridis'); ax[1].set_title("Occlusion (raw drop)")
        ax[1].axis('off'); plt.colorbar(im, ax=ax[1], fraction=0.046, label='Δp')
        ax[2].imshow(heat_full, cmap='viridis'); ax[2].set_title("Upsampled"); ax[2].axis('off')
        ax[3].imshow(self._overlay(base, heat_full)); ax[3].set_title("Overlay"); ax[3].axis('off')
        plt.suptitle(f'Occlusion / Glimpse — baseline p={orig_prob:.4f}, '
                     f'patch={patch_size}, stride={stride}', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'occlusion_result.png'), dpi=150)
        plt.close()
        return heat_full

    def explain_lime(self, img_tensor, target_class=None):
        explainer = lime_image.LimeImageExplainer(verbose=False)
        img_np = self._to_display(img_tensor).astype(np.float64)

        def predict_fn(images):
            self.model.eval()
            tensors = [HierarchicalCancerDataset.preprocess_from_hwc_float(
                np.asarray(im, dtype=np.float32)) for im in images]
            batch = torch.stack(tensors)
            return self._batched_stage_probs(batch, chunk=64)

        explanation = explainer.explain_instance(
            img_np, predict_fn,
            top_labels=3, hide_color=None,
            num_samples=self.config.lime_num_samples,
            batch_size=32
        )
        label = target_class if (target_class in explanation.local_exp) else explanation.top_labels[0]

        temp_p, mask_p = explanation.get_image_and_mask(
            label, positive_only=True, num_features=8, hide_rest=False)
        temp_all, mask_all = explanation.get_image_and_mask(
            label, positive_only=False, num_features=12, hide_rest=False)

        seg = explanation.segments
        weights = dict(explanation.local_exp[label])
        weight_map = np.zeros(seg.shape, dtype=np.float32)
        for sid, w in weights.items():
            weight_map[seg == sid] = w
        vmax = np.abs(weight_map).max() + 1e-8

        fig, ax = plt.subplots(1, 4, figsize=(17, 4.3))
        ax[0].imshow(img_np); ax[0].set_title('Input'); ax[0].axis('off')
        ax[1].imshow(mark_boundaries(temp_p, mask_p))
        ax[1].set_title('LIME — supporting regions'); ax[1].axis('off')
        ax[2].imshow(mark_boundaries(temp_all, mask_all))
        ax[2].set_title('LIME — pro & contra'); ax[2].axis('off')
        im = ax[3].imshow(weight_map, cmap='coolwarm', vmin=-vmax, vmax=vmax)
        ax[3].set_title('LIME weight map'); ax[3].axis('off')
        plt.colorbar(im, ax=ax[3], fraction=0.046)
        plt.suptitle(f'LIME — class: {self.stage_names[label]} '
                     f'({self.config.lime_num_samples} samples)', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'lime_result.png'), dpi=150)
        plt.close()
        return weight_map

    def explain_shap(self, img_tensor, target_class=0):
        img_np = self._to_display(img_tensor).astype(np.float32)
        segments = slic(img_np, n_segments=self.config.shap_segments,
                        compactness=10, start_label=0)
        seg_ids = np.unique(segments)
        num_segments = len(seg_ids)
        seg_index = {s: i for i, s in enumerate(seg_ids)}
        fill_value = img_np.reshape(-1, 3).mean(axis=0)

        def mask_model_runner(masks):
            tensors = []
            for mask in masks:
                temp = img_np.copy()
                for s in seg_ids:
                    if mask[seg_index[s]] == 0:
                        temp[segments == s] = fill_value
                tensors.append(HierarchicalCancerDataset.preprocess_from_hwc_float(temp))
            batch = torch.stack(tensors)
            return self._batched_stage_probs(batch, chunk=64)

        explainer = shap.KernelExplainer(mask_model_runner, np.zeros((1, num_segments)))
        shap_values = explainer.shap_values(np.ones((1, num_segments)),
                                            nsamples=self.config.shap_nsamples,
                                            silent=True)

        if isinstance(shap_values, list):
            sv = np.asarray(shap_values[target_class])[0]
        else:
            arr = np.asarray(shap_values)
            if arr.ndim == 3:
                sv = arr[0, :, target_class]
            elif arr.ndim == 2:
                sv = arr[0]
            else:
                sv = arr.ravel()[:num_segments]
        sv = np.asarray(sv).ravel()[:num_segments]

        shap_img = np.zeros(segments.shape, dtype=np.float32)
        for s in seg_ids:
            shap_img[segments == s] = sv[seg_index[s]]
        vmax = np.abs(shap_img).max() + 1e-8

        order = np.argsort(-np.abs(sv))[:10]
        top_mask = np.zeros(segments.shape, dtype=bool)
        for k in order[:5]:
            top_mask |= (segments == seg_ids[k])

        fig, ax = plt.subplots(1, 4, figsize=(18, 4.5))
        ax[0].imshow(img_np); ax[0].set_title("Input"); ax[0].axis('off')
        ax[1].imshow(mark_boundaries(img_np, segments))
        ax[1].set_title(f"SLIC superpixels (n={num_segments})"); ax[1].axis('off')
        im = ax[2].imshow(shap_img, cmap='coolwarm', vmin=-vmax, vmax=vmax)
        ax[2].set_title("SHAP attribution"); ax[2].axis('off')
        plt.colorbar(im, ax=ax[2], fraction=0.046)
        ax[3].imshow(mark_boundaries(img_np, top_mask.astype(int)))
        ax[3].set_title("Top-5 |SHAP| regions"); ax[3].axis('off')
        plt.suptitle(f'Kernel SHAP — class {self.stage_names[target_class]} '
                     f'({self.config.shap_nsamples} coalitions)', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'shap_result.png'), dpi=150)
        plt.close()

        plt.figure(figsize=(8, 5))
        vals = sv[order][::-1]
        colors = ['#c0392b' if v > 0 else '#2471a3' for v in vals]
        plt.barh([f'SP {seg_ids[k]}' for k in order][::-1], vals, color=colors)
        plt.xlabel('SHAP value'); plt.title('Top-10 superpixel contributions')
        plt.axvline(0, color='k', lw=.8)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'shap_bar.png'), dpi=150)
        plt.close()
        return shap_img

    def explain_consensus_panel(self, img_tensor, target_class):
        base = self._to_display(img_tensor)
        maps = {}
        try: maps['Grad-CAM'] = self.explain_gradcam(img_tensor, target_class)
        except Exception as e: print(f"  [consensus] Grad-CAM skipped: {e}")
        try: maps['Grad-CAM++'] = self.explain_gradcam_plusplus(img_tensor, target_class)
        except Exception as e: print(f"  [consensus] Grad-CAM++ skipped: {e}")
        try: maps['Occlusion'] = self.explain_occlusion(img_tensor, target_class)
        except Exception as e: print(f"  [consensus] Occlusion skipped: {e}")
        try: maps['IntegratedGrad'] = self.explain_integrated_gradients(img_tensor, target_class)
        except Exception as e: print(f"  [consensus] IG skipped: {e}")

        if len(maps) < 2:
            return

        names = list(maps.keys())
        n = len(names)
        fig, ax = plt.subplots(1, n + 1, figsize=(4.2 * (n + 1), 4.2))
        ax[0].imshow(base); ax[0].set_title('Input'); ax[0].axis('off')
        for i, k in enumerate(names):
            ax[i + 1].imshow(self._overlay(base, self._norm01(maps[k])))
            ax[i + 1].set_title(k); ax[i + 1].axis('off')
        plt.suptitle(f'XAI consensus — {self.stage_names[target_class]}', fontsize=12)
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'xai_consensus_panel.png'), dpi=150)
        plt.close()

        from scipy.stats import spearmanr
        C = np.eye(n)
        for i in range(n):
            for j in range(i + 1, n):
                a = self._norm01(maps[names[i]]).ravel()
                b = self._norm01(maps[names[j]]).ravel()
                r = spearmanr(a, b).correlation
                C[i, j] = C[j, i] = 0.0 if np.isnan(r) else r
        plt.figure(figsize=(6.5, 5.5))
        sns.heatmap(C, annot=True, fmt='.3f', cmap='RdYlGn', vmin=-1, vmax=1,
                    xticklabels=names, yticklabels=names)
        plt.title('Spearman agreement between XAI methods')
        plt.tight_layout()
        plt.savefig(os.path.join(self.dirs['xai'], 'xai_agreement_matrix.png'), dpi=150)
        plt.close()


# ==========================================
# 7. Main Execution Pipeline (70 / 15 / 15 Split)
# ==========================================
def build_grouped_split_3way(samples, train_frac, val_frac, test_frac, seed, patient_aware=True):
    assert abs((train_frac + val_frac + test_frac) - 1.0) < 1e-5, "Fractions must sum to 1.0"

    if not patient_aware:
        rng = np.random.RandomState(seed)
        idx = rng.permutation(len(samples))
        n_train = int(len(samples) * train_frac)
        n_val = int(len(samples) * val_frac)
        train_idx = idx[:n_train].tolist()
        val_idx = idx[n_train:n_train + n_val].tolist()
        test_idx = idx[n_train + n_val:].tolist()
        return sorted(train_idx), sorted(val_idx), sorted(test_idx)

    groups = defaultdict(list)
    for i, s in enumerate(samples):
        groups[s['patient_id']].append(i)

    stage_of_group = {g: samples[ix[0]]['stage_name'] for g, ix in groups.items()}
    by_stage = defaultdict(list)
    for g, st in stage_of_group.items():
        by_stage[st].append(g)

    train_idx, val_idx, test_idx = [], [], []
    for st, glist in by_stage.items():
        glist = sorted(glist)
        rng = np.random.RandomState(seed + (hash(st) % 10000))
        rng.shuffle(glist)
        
        n_total = len(glist)
        n_val = max(1, int(round(n_total * val_frac))) if n_total > 2 else 0
        n_test = max(1, int(round(n_total * test_frac))) if n_total > 2 else 0
        
        if n_val + n_test >= n_total and n_total >= 3:
            n_val = 1
            n_test = 1
        
        val_groups = glist[:n_val]
        test_groups = glist[n_val:n_val + n_test]
        train_groups = glist[n_val + n_test:]

        if not train_groups and val_groups:
            train_groups.append(val_groups.pop())

        for g in val_groups:
            val_idx.extend(groups[g])
        for g in test_groups:
            test_idx.extend(groups[g])
        for g in train_groups:
            train_idx.extend(groups[g])

    return sorted(train_idx), sorted(val_idx), sorted(test_idx)


def main():
    config = Config()

    raw_dirs = get_raw_dataset_paths()

    train_dataset = HierarchicalCancerDataset(data_dirs=raw_dirs, config=config, is_train=True)
    shared_index = train_dataset.export_index()
    val_test_dataset = HierarchicalCancerDataset(data_dirs=raw_dirs, config=config,
                                                 is_train=False, shared_index=shared_index)

    train_indices, val_indices, test_indices = build_grouped_split_3way(
        shared_index['samples'],
        train_frac=config.train_fraction,
        val_frac=config.val_fraction,
        test_frac=config.test_fraction,
        seed=config.split_seed,
        patient_aware=config.patient_aware_split
    )
    print(f"[Split 3-Way] train={len(train_indices)}  val={len(val_indices)}  test={len(test_indices)}  "
          f"patient_aware={config.patient_aware_split}")

    train_ds = torch.utils.data.Subset(train_dataset, train_indices)
    val_ds = torch.utils.data.Subset(val_test_dataset, val_indices)
    test_ds = torch.utils.data.Subset(val_test_dataset, test_indices)

    train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,
                              pin_memory=True, num_workers=2, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False,
                            pin_memory=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False,
                             pin_memory=True, num_workers=2)

    model = CanceRX(config)
    criterion = OmniLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=config.lr,
        weight_decay=config.weight_decay, 
        eps=1e-5
    )

    analyzer = ComplexityAnalyzer(model, config)
    analyzer.run(os.path.join(config.output_dir, 'complexity'))

    print(f"\n--- Training CanceRX: 2-Level Multi-Classification Pipeline (70/15/15 Split) ---")
    trainer = Trainer(model, train_loader, val_loader, criterion, optimizer, config)
    trainer.train()

    best_path = os.path.join(config.ckpt_dir, 'mtpg_net_best.pth')
    if os.path.exists(best_path):
        ck = torch.load(best_path, map_location=config.device)
        model.load_state_dict(ck['model_state_dict'])
        print(f"[Eval] Loaded best checkpoint (epoch {ck['epoch']}) for unseen TEST evaluation.")

    inferencer = Inferencer(model, test_loader, shared_index['types'], shared_index['stages'], config)
    inferencer.generate_all_explainability_artifacts()


if __name__ == '__main__':
    main()